# Regressor only tasks

## Imports and constants

In [2]:
import dsi #De Simone Inference module
import itertools
import time
import importlib
import openml
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_openml

RANDOM_STATE = int('8'*8)
DARK_MODE = True
VAL_SIZE = 0.15
TEST_SIZE = 15/85

In [3]:
dsi.is_gpu_backend() #Working on Gpu control

True

## Routines functions

In [31]:
def load_dataset_from_task(task_id):
    """
    Load a CTR23 dataset using its OpenML task ID.
    Returns df and target column name.
    """

    # Load the task
    task = openml.tasks.get_task(task_id)

    # Extract the dataset associated with the task
    dataset = task.get_dataset()

    # Identify the target column
    target_name = dataset.default_target_attribute

    # Load data as pandas DataFrame
    X, y, _, _ = dataset.get_data(
        target=target_name,
        dataset_format="dataframe"
    )

    # Build final DataFrame
    X[target_name] = y
    
    return X, target_name


def split_convert_dataset(df, shuffle=True):
    """
    Splits the dataset and converts to array-like float64.
    Returns train, validation and test arrays.
    """
    
    df_train, df_test = train_test_split(
        df,
        test_size=TEST_SIZE,
        shuffle=shuffle,
        random_state=RANDOM_STATE,
    )

    df_train, df_val = train_test_split(
        df_train,
        test_size=VAL_SIZE,
        shuffle=shuffle,
        random_state=RANDOM_STATE,
    )
    
    # Convert to numpy float64
    train_arr = df_train.to_numpy(dtype=np.float64)
    val_arr   = df_val.to_numpy(dtype=np.float64)
    test_arr  = df_test.to_numpy(dtype=np.float64)

    return train_arr, val_arr, test_arr


def collapse_rare_categories(df, max_levels=1000):
    df = df.copy()
    cat_cols = df.select_dtypes(include=["object", "category"]).columns

    for col in cat_cols:
        
        #Introduce a special categorywhere data is missing simil to ctr23 pipeline
        df[col] = df[col].fillna("__MISSING__")
        
        counts = df[col].value_counts()

        if len(counts) <= max_levels:
            continue

        keep = counts.nlargest(max_levels).index
        df[col] = df[col].where(df[col].isin(keep), "__OTHER__")

        df[col] = df[col].astype("category")

    return df


def one_hot_encode(df):
    df = df.copy()
    cat_cols = df.select_dtypes(include=["object", "category"]).columns

    if len(cat_cols) == 0:
        return df

    df[cat_cols] = df[cat_cols].fillna("__MISSING__")

    encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    encoded = encoder.fit_transform(df[cat_cols])

    encoded_df = (
        pd.DataFrame(encoded, columns=encoder.get_feature_names_out(cat_cols))
        .astype("int8")
        .reset_index(drop=True)
    )

    df = df.drop(columns=cat_cols).reset_index(drop=True)

    return pd.concat([df, encoded_df], axis=1)


def preprocess_ctr23_style(df, max_levels=1000):
    df = df.copy()

    # Save original categorical columns
    original_cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

    df = collapse_rare_categories(df, max_levels=max_levels)
    df = one_hot_encode(df)

    return df, original_cat_cols


def move_target_last(df, target_name):
    target_col = df[target_name]
    df = df.drop(columns=[target_name])
    df[target_name] = target_col
    return df


def generate_unique_list(df_encoded, original_cat_cols):
    unique_list = []

    for col in df_encoded.columns:
        # If the column name starts with any original categorical column name -> it's one-hot
        is_one_hot = any(col.startswith(cat + "_") for cat in original_cat_cols)
        unique_list.append(is_one_hot)

    return unique_list


def load_preprocess_split(task_id):
    
    # Load dataset
    df_raw, target_name = load_dataset_from_task(task_id)

    # Preprocess CTR23-style
    df_encoded, original_cat_cols = preprocess_ctr23_style(df_raw)

    # Move target to last column (your optimized idea)
    df_encoded = move_target_last(df_encoded, target_name)

    # Generate unique_list
    unique_list = generate_unique_list(df_encoded, original_cat_cols)

    # Split e convert
    train_arr, val_arr, test_arr = split_convert_dataset(df_encoded)

    return train_arr, val_arr, test_arr, unique_list


def generate_configurations(unique_list):
    """
    Generate all 8 combinations of DSI flags.

    parameters
    ----------
    unique_list: list
        List of bool: True where operator forces the division of feature domain
        on the unique values contained. False where the approach follows density
        based or linear feature domain division.

    returns
    -------
    configs: dict
        Various configuration to test the model.
    """

    configs = []

    fire_options = [True, False]
    fuzzy_options = [True, False]
    db_options = [True, False]

    for fire, fuzzy, db in itertools.product(fire_options, fuzzy_options, db_options):

        # Base division depending on DB flag
        if db:
            division = ["D"] * len(unique_list)
        else:
            division = ["L"] * len(unique_list)

        # Enforce class-based division
        for i, is_unique in enumerate(unique_list):
            if is_unique:
                division[i] = "U"

        configs.append({
            "fire_of_train_rules": fire,
            "fuzzy_output": fuzzy,
            "default_division": division,
        })

    return configs


def regression_test(df_train, df_val, df_test, configurations):
    """
    Test all 8 configuration for regression task solved by De Simone Inference
    """
    results = []
    
    for config in configurations:

        print("Starting optimizzation for:\n",config)
        
        model = dsi.DSregressor(**config)
        model.fit_cross_validation(df_train, df_val, domains_functions_list=config["default_division"])

        # --- Predizione ---
        y_out = model.predict(df_test[:, :-1])
        y_true = df_test[:, -1]
        y_pred = y_out.get()

        # --- Gestione predizioni mancanti ---
        valid_mask = ~np.isnan(y_pred)
        cov = valid_mask.mean()   # coverage

        if valid_mask.sum() == 0:
            y_pred_valid = np.array([])
            y_true_valid = np.array([])
        else:
            y_pred_valid = y_pred[valid_mask]
            y_true_valid = y_true[valid_mask]

        # --- Entry base ---
        entry = {
            "fire_of_train_rules": config["fire_of_train_rules"],
            "fuzzy_output": config["fuzzy_output"],
            "default_division": config["default_division"],
            "cov": cov
        }

        # --- Score ---
        if len(y_pred_valid) > 0:
            #entry["R2"] = dsi.r2(y_true_valid, y_pred_valid).get()
            #entry["MAE"] = dsi.mae(y_true_valid, y_pred_valid).get()
            entry["RMSE"] = dsi.rmse(y_true_valid, y_pred_valid).get()
        else:
            #entry["r2"] = np.nan
            #entry["MAE"] = np.nan
            entry["RMSE"] = np.nan

        results.append(entry)

    return results

def scoring_routine(task_id):
    """
    The whole scoring routines, outputs scoring results
    """
    df_train, df_val, df_test, unique_list = load_preprocess_split(task_id)
    configs = generate_configurations(unique_list)
    return regression_test(df_train, df_val, df_test, configs)

## Test on datasets

In [32]:
ctr23_task_ids = [
    361234, 361235, 361236, 361237, 361241, 361242, 361243, 361244,
    361247, 361249, 361250, 361251, 361252, 361253, 361254, 361255,
    361256, 361257, 361258, 361259, 361260, 361261, 361264, 361266,
    361267, 361268, 361269, 361272, 361616, 361617, 361618, 361619,
    361621, 361622, 361623
]

In [26]:
df_train, df_val, df_test, unique_list = load_preprocess_split(361234)

In [7]:
configs = generate_configurations(unique_list)

In [8]:
returns = regression_test(df_train, df_val, df_test)

Starting optimizzation for:
 {'fire_of_train_rules': True, 'fuzzy_output': True, 'default_division': ['D', 'D', 'D', 'D', 'D', 'D', 'D', 'U', 'U', 'U', 'D']}
|----> Starting Global Optimization 
|----> New best score with n_of_sets: [5 5 5 5 5 5 5 2 2 2 5] -> Scored: 0.3210843681803537
|----> New best score with n_of_sets: [6 6 6 6 6 6 6 2 2 2 6] -> Scored: 0.35654877672717605
|--------> Starting Coordinate-wise Optimization 
|--------> New best score with n_of_sets: [7 6 6 6 6 6 6 2 2 2 6] -> Scored: 0.374410047783002
|--------> New best score with n_of_sets: [7 6 6 6 6 5 6 2 2 2 6] -> Scored: 0.3758197507771782
|--------> New best score with n_of_sets: [7 6 6 6 6 5 5 2 2 2 6] -> Scored: 0.37691801178366263
|--------> New best score with n_of_sets: [7 6 6 6 6 5 5 2 2 2 7] -> Scored: 0.3860526922071835
|--------> New best score with n_of_sets: [7 5 6 6 6 5 5 2 2 2 7] -> Scored: 0.3929615072224024
|------------> Cross-validation ended succesfully:
|------------> [7 5 6 6 6 5 5 2 2 2 7] 

C:\Users\Loris\Desktop\git-hub\DSInference\src\dsi\domainDivision.py:103: RuntimeWarning: Domain division failed: insufficient unique values in the feature domain caused repeated quantile pivots. The effective number of fuzzy sets is now lower than the theoretical. Consider using linear_division or unique_division for this feature.
  centers = _check_for_unique_ordered_centers(xp.quantile(arr, quantiles)) # Centers


|--------> New best score with n_of_sets: [7 6 7 6 8 7 7 2 2 2 8] -> Scored: 0.4522830290191471
|--------> New best score with n_of_sets: [7 6 7 6 8 7 8 2 2 2 8] -> Scored: 0.45737915629801307
|--------> New best score with n_of_sets: [7 5 7 6 8 7 8 2 2 2 8] -> Scored: 0.4626459182389137
|--------> New best score with n_of_sets: [7 5 7 6 8 7 8 2 2 2 7] -> Scored: 0.4650821078524771
|------------> Cross-validation ended succesfully:
|------------> [7 5 7 6 8 7 8 2 2 2 7] -> Best number of set per feature
|------------> 0.4650821078524771 -> Best score: r2 metric multiplied by coverage
|------------> 16.04401993751526s -> Optimization total time, of wich
|------------> 0.9018363952636719s -> First step and 15.142183542251587s -> Second step 
|------------> 0.1487882055085281s -> Mean training time for different configuration. 

Starting optimizzation for:
 {'fire_of_train_rules': True, 'fuzzy_output': False, 'default_division': ['L', 'L', 'L', 'L', 'L', 'L', 'L', 'U', 'U', 'U', 'L']}
|--

In [9]:
returns

[{'fire_of_train_rules': True,
  'fuzzy_output': True,
  'default_division': ['D', 'D', 'D', 'D', 'D', 'D', 'D', 'U', 'U', 'U', 'D'],
  'cov': np.float64(0.9959349593495935),
  'RMSE': array(2.55381546)},
 {'fire_of_train_rules': True,
  'fuzzy_output': True,
  'default_division': ['L', 'L', 'L', 'L', 'L', 'L', 'L', 'U', 'U', 'U', 'L'],
  'cov': np.float64(0.994579945799458),
  'RMSE': array(2.37435689)},
 {'fire_of_train_rules': True,
  'fuzzy_output': False,
  'default_division': ['D', 'D', 'D', 'D', 'D', 'D', 'D', 'U', 'U', 'U', 'D'],
  'cov': np.float64(0.9932249322493225),
  'RMSE': array(2.48576341)},
 {'fire_of_train_rules': True,
  'fuzzy_output': False,
  'default_division': ['L', 'L', 'L', 'L', 'L', 'L', 'L', 'U', 'U', 'U', 'L'],
  'cov': np.float64(0.997289972899729),
  'RMSE': array(2.32506569)},
 {'fire_of_train_rules': False,
  'fuzzy_output': True,
  'default_division': ['D', 'D', 'D', 'D', 'D', 'D', 'D', 'U', 'U', 'U', 'D'],
  'cov': np.float64(0.994579945799458),
  'RM

In [33]:
returns_airfoil = scoring_routine(361235)

Starting optimizzation for:
 {'fire_of_train_rules': True, 'fuzzy_output': True, 'default_division': ['D', 'D', 'D', 'D', 'D', 'D']}


C:\Users\Loris\Desktop\git-hub\DSInference\src\dsi\domainDivision.py:103: RuntimeWarning: Domain division failed: insufficient unique values in the feature domain caused repeated quantile pivots. The effective number of fuzzy sets is now lower than the theoretical. Consider using linear_division or unique_division for this feature.
  centers = _check_for_unique_ordered_centers(xp.quantile(arr, quantiles)) # Centers


|----> Starting Global Optimization 
|----> New best score with n_of_sets: [5 5 5 5 5 5] -> Scored: 0.7525518012187024
|--------> Starting Coordinate-wise Optimization 
|--------> New best score with n_of_sets: [6 5 5 5 5 5] -> Scored: 0.7624609296078759
|--------> New best score with n_of_sets: [6 4 5 5 5 5] -> Scored: 0.7751631176344583
|--------> New best score with n_of_sets: [6 4 5 5 5 6] -> Scored: 0.7910542641727049
|------------> Cross-validation ended succesfully:
|------------> [6 4 5 5 5 6] -> Best number of set per feature
|------------> 0.7910542641727049 -> Best score: r2 metric multiplied by coverage
|------------> 3.3869307041168213s -> Optimization total time, of wich
|------------> 0.301738977432251s -> First step and 3.0851917266845703s -> Second step 
|------------> 0.11480832099914551s -> Mean training time for different configuration. 

Starting optimizzation for:
 {'fire_of_train_rules': True, 'fuzzy_output': True, 'default_division': ['L', 'L', 'L', 'L', 'L', 'L

In [40]:
returns_airfoil

[{'fire_of_train_rules': True,
  'fuzzy_output': True,
  'default_division': ['D', 'D', 'D', 'D', 'D', 'D'],
  'cov': np.float64(0.9962406015037594),
  'RMSE': array(3.40795275)},
 {'fire_of_train_rules': True,
  'fuzzy_output': True,
  'default_division': ['L', 'L', 'L', 'L', 'L', 'L'],
  'cov': np.float64(0.9699248120300752),
  'RMSE': array(3.4222034)},
 {'fire_of_train_rules': True,
  'fuzzy_output': False,
  'default_division': ['D', 'D', 'D', 'D', 'D', 'D'],
  'cov': np.float64(0.9962406015037594),
  'RMSE': array(3.54125641)},
 {'fire_of_train_rules': True,
  'fuzzy_output': False,
  'default_division': ['L', 'L', 'L', 'L', 'L', 'L'],
  'cov': np.float64(0.9849624060150376),
  'RMSE': array(4.01223277)},
 {'fire_of_train_rules': False,
  'fuzzy_output': True,
  'default_division': ['D', 'D', 'D', 'D', 'D', 'D'],
  'cov': np.float64(0.9962406015037594),
  'RMSE': array(3.39715365)},
 {'fire_of_train_rules': False,
  'fuzzy_output': True,
  'default_division': ['L', 'L', 'L', 'L',

In [41]:
returns_3 = scoring_routine(361236)

Starting optimizzation for:
 {'fire_of_train_rules': True, 'fuzzy_output': True, 'default_division': ['D', 'D', 'D', 'D', 'D', 'U', 'U', 'U', 'U', 'U', 'U', 'U', 'U', 'U', 'U', 'U', 'D']}
|----> Starting Global Optimization 
|----> New best score with n_of_sets: [5 5 5 5 5 2 2 2 2 2 2 2 2 2 2 2 5] -> Scored: 0.5733904389917753


C:\Users\Loris\Desktop\git-hub\DSInference\src\dsi\domainDivision.py:103: RuntimeWarning: Domain division failed: insufficient unique values in the feature domain caused repeated quantile pivots. The effective number of fuzzy sets is now lower than the theoretical. Consider using linear_division or unique_division for this feature.
  centers = _check_for_unique_ordered_centers(xp.quantile(arr, quantiles)) # Centers


|----> New best score with n_of_sets: [6 6 6 6 6 2 2 2 2 2 2 2 2 2 2 2 6] -> Scored: 0.6513469930653375
|--------> Starting Coordinate-wise Optimization 
|--------> New best score with n_of_sets: [6 6 6 6 6 2 2 2 2 2 2 2 2 2 2 2 7] -> Scored: 0.6698151790387794
|------------> Cross-validation ended succesfully:
|------------> [6 6 6 6 6 2 2 2 2 2 2 2 2 2 2 2 7] -> Best number of set per feature
|------------> 0.6698151790387794 -> Best score: r2 metric multiplied by coverage
|------------> 3.998975992202759s -> Optimization total time, of wich
|------------> 0.5753319263458252s -> First step and 3.4236440658569336s -> Second step 
|------------> 0.11685423193306758s -> Mean training time for different configuration. 

Starting optimizzation for:
 {'fire_of_train_rules': True, 'fuzzy_output': True, 'default_division': ['L', 'L', 'L', 'L', 'L', 'U', 'U', 'U', 'U', 'U', 'U', 'U', 'U', 'U', 'U', 'U', 'L']}
|----> Starting Global Optimization 
|----> New best score with n_of_sets: [5 5 5 5 

In [44]:
returns_3

[{'fire_of_train_rules': True,
  'fuzzy_output': True,
  'default_division': ['D',
   'D',
   'D',
   'D',
   'D',
   'U',
   'U',
   'U',
   'U',
   'U',
   'U',
   'U',
   'U',
   'U',
   'U',
   'U',
   'D'],
  'cov': np.float64(0.7867036011080333),
  'RMSE': array(4373.56825098)},
 {'fire_of_train_rules': True,
  'fuzzy_output': True,
  'default_division': ['L',
   'L',
   'L',
   'L',
   'L',
   'U',
   'U',
   'U',
   'U',
   'U',
   'U',
   'U',
   'U',
   'U',
   'U',
   'U',
   'L'],
  'cov': np.float64(0.7977839335180056),
  'RMSE': array(3592.91515237)},
 {'fire_of_train_rules': True,
  'fuzzy_output': False,
  'default_division': ['D',
   'D',
   'D',
   'D',
   'D',
   'U',
   'U',
   'U',
   'U',
   'U',
   'U',
   'U',
   'U',
   'U',
   'U',
   'U',
   'D'],
  'cov': np.float64(0.7977839335180056),
  'RMSE': array(3248.28904432)},
 {'fire_of_train_rules': True,
  'fuzzy_output': False,
  'default_division': ['L',
   'L',
   'L',
   'L',
   'L',
   'U',
   'U',
   'U',
 

In [45]:
returns_energy = scoring_routine(361250)

Starting optimizzation for:
 {'fire_of_train_rules': True, 'fuzzy_output': True, 'default_division': ['D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D']}
|----> Starting Global Optimization 
|----> New best score with n_of_sets: [5 5 5 5 5 5 5 5 5 5 5 5] -> Scored: -0.003210345607415066


C:\Users\Loris\Desktop\git-hub\DSInference\src\dsi\domainDivision.py:103: RuntimeWarning: Domain division failed: insufficient unique values in the feature domain caused repeated quantile pivots. The effective number of fuzzy sets is now lower than the theoretical. Consider using linear_division or unique_division for this feature.
  centers = _check_for_unique_ordered_centers(xp.quantile(arr, quantiles)) # Centers


|----> New best score with n_of_sets: [4 4 4 4 4 4 4 4 4 4 4 4] -> Scored: 0.21336361450186614
|--------> Starting Coordinate-wise Optimization 
|--------> New best score with n_of_sets: [4 5 4 4 4 4 4 4 4 4 4 4] -> Scored: 0.23586961458672942
|--------> New best score with n_of_sets: [4 5 4 3 4 4 4 4 4 4 4 4] -> Scored: 0.24483318553651778
|--------> New best score with n_of_sets: [4 5 4 3 3 4 4 4 4 4 4 4] -> Scored: 0.27655083467560726
|--------> New best score with n_of_sets: [4 5 4 3 3 4 5 4 4 4 4 4] -> Scored: 0.2881543700395225
|--------> New best score with n_of_sets: [4 5 4 3 3 4 5 3 4 4 4 4] -> Scored: 0.29468868417246097
|--------> New best score with n_of_sets: [4 5 4 3 3 4 5 3 5 4 4 4] -> Scored: 0.30693048539794726
|--------> New best score with n_of_sets: [4 5 4 3 3 4 6 3 5 4 4 4] -> Scored: 0.32315677772477647
|------------> Cross-validation ended succesfully:
|------------> [4 5 4 3 3 4 6 3 5 4 4 4] -> Best number of set per feature
|------------> 0.32315677772477647 ->

In [46]:
returns_energy

[{'fire_of_train_rules': True,
  'fuzzy_output': True,
  'default_division': ['D',
   'D',
   'D',
   'D',
   'D',
   'D',
   'D',
   'D',
   'D',
   'D',
   'D',
   'D'],
  'cov': np.float64(0.9893992932862191),
  'RMSE': array(0.67734955)},
 {'fire_of_train_rules': True,
  'fuzzy_output': True,
  'default_division': ['L',
   'L',
   'L',
   'L',
   'L',
   'L',
   'L',
   'L',
   'L',
   'L',
   'L',
   'L'],
  'cov': np.float64(0.9540636042402827),
  'RMSE': array(0.628989)},
 {'fire_of_train_rules': True,
  'fuzzy_output': False,
  'default_division': ['D',
   'D',
   'D',
   'D',
   'D',
   'D',
   'D',
   'D',
   'D',
   'D',
   'D',
   'D'],
  'cov': np.float64(0.9964664310954063),
  'RMSE': array(0.68125998)},
 {'fire_of_train_rules': True,
  'fuzzy_output': False,
  'default_division': ['L',
   'L',
   'L',
   'L',
   'L',
   'L',
   'L',
   'L',
   'L',
   'L',
   'L',
   'L'],
  'cov': np.float64(0.9717314487632509),
  'RMSE': array(0.64167659)},
 {'fire_of_train_rules': Fal